In [1]:
import sys
import os
import re
import json
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime

import pandas as pd
from tqdm import tqdm

PROJECT_ROOT = Path("../../").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from lib.llm.llm_client import MODEL, MODEL_N, timed_completion, strip_thinking

In [2]:
DATASETS_ROOT = PROJECT_ROOT / "datasets" / "experiments" / "agent" / "tasks"

DIVISION_DIR  = DATASETS_ROOT / "division"
EXTRACTION_DIR = DATASETS_ROOT / "extraction"
META_DIR      = DATASETS_ROOT / "meta"

In [3]:
# Cell A — Load latest CSVs
def load_latest_csv(directory: Path) -> pd.DataFrame:
    csvs = sorted(directory.glob("*.csv"))
    if not csvs:
        raise FileNotFoundError(f"No CSVs in {directory}")
    latest = csvs[-1]
    print(f"Loading: {latest.name}")
    return pd.read_csv(latest)

df_division   = load_latest_csv(DIVISION_DIR)
df_extraction = load_latest_csv(EXTRACTION_DIR)
df_meta       = load_latest_csv(META_DIR)

print(f"Division:   {len(df_division)} rows")
print(f"Extraction: {len(df_extraction)} rows")
print(f"Meta:       {len(df_meta)} rows")

Loading: division_20260624_143048.csv
Loading: extraction_20260624_143048.csv
Loading: meta_20260624_143048.csv
Division:   32 rows
Extraction: 218 rows
Meta:       32 rows


In [4]:
# Cell B — Tool definitions (operate on dataframes)

def search_extractions(company: str = None, keyword: str = None,
                       extract_type: str = None, top_k: int = 10) -> list[dict]:
    df = df_extraction.copy()
    if company:
        df = df[df["document"].str.contains(company, case=False, na=False)]
    if keyword:
        mask = (df["content"].str.contains(keyword, case=False, na=False) |
                df["title"].str.contains(keyword, case=False, na=False))
        df = df[mask]
    if extract_type:
        df = df[df["type"].str.contains(extract_type, case=False, na=False)]
    df = df.sort_values("relevance_score", ascending=False).head(top_k)
    return df[["document", "page_range", "title", "content", "type", "relevance_score", "original_chunk"]].to_dict("records")

def get_original_chunk(document: str, chunk_idx: int) -> dict:
    row = df_extraction[
        df_extraction["document"].str.contains(document, case=False, na=False) &
        (df_extraction["chunk_idx"] == chunk_idx)
    ]
    if row.empty:
        return {"error": "Not found"}
    return row[["document", "chunk_idx", "page_range", "original_chunk"]].iloc[0].to_dict()

def search_by_relevance(company: str, min_score: float = 0.7) -> list[dict]:
    df = df_extraction[
        df_extraction["document"].str.contains(company, case=False, na=False) &
        (df_extraction["relevance_score"] >= min_score)
    ].sort_values("relevance_score", ascending=False)
    return df[["document", "page_range", "title", "content", "type", "relevance_score"]].to_dict("records")

def get_document_metadata(company: str) -> list[dict]:
    df = df_meta[df_meta["document"].str.contains(company, case=False, na=False)]
    return df[["document", "index", "total_pages", "num_chunks", "num_extractions", "strategy"]].to_dict("records")

TOOLS = {
    "search_extractions":    search_extractions,
    "get_original_chunk":    get_original_chunk,
    "search_by_relevance":   search_by_relevance,
    "get_document_metadata": get_document_metadata,
}

TOOLS_SCHEMA = """
Available tools (call as JSON):
1. search_extractions(company, keyword, extract_type, top_k)
2. get_original_chunk(document, chunk_idx)
3. search_by_relevance(company, min_score)
4. get_document_metadata(company)
"""

In [5]:
# Cell C — Sub-agent (ReAct loop)

SUB_AGENT_SYSTEM = """You are a financial document research agent. Answer questions by calling tools to search extracted financial document chunks.

{tools_schema}

At each step output ONLY valid JSON in one of these formats:

To call a tool:
{{"action": "tool", "tool": "tool_name", "args": {{...}}}}

When done:
{{"action": "answer", "answer": "precise answer", "sources": [{{"document": "...", "page_range": "...", "extract": "..."}}]}}

Be precise. Cite sources. Use multiple tools."""

def run_sub_agent(question: str, company: str, strategy_hint: str, max_steps: int = 6) -> dict:
    system = SUB_AGENT_SYSTEM.format(tools_schema=TOOLS_SCHEMA)
    messages = [
        {"role": "system", "content": system},
        {"role": "user",   "content": f"Company: {company}\nStrategy: {strategy_hint}\nQuestion: {question}\n\nBegin."}
    ]

    tools_used = []
    steps = 0

    while steps < max_steps:
        response, latency_ms = timed_completion(
            messages, n=MODEL_N,
            extra_body={"chat_template_kwargs": {"enable_thinking": False}} if MODEL_N == 1 else {}
        )
        raw = strip_thinking(response.choices[0].message.content)
        raw = re.sub(r"```json|```", "", raw).strip()

        try:
            parsed = json.loads(raw)
        except json.JSONDecodeError:
            match = re.search(r'\{.*\}', raw, re.DOTALL)
            parsed = json.loads(match.group()) if match else None
            if not parsed:
                break

        messages.append({"role": "assistant", "content": raw})

        if parsed.get("action") == "answer":
            return {
                "answer":     parsed.get("answer", ""),
                "sources":    parsed.get("sources", []),
                "tools_used": tools_used,
                "steps":      steps,
            }

        elif parsed.get("action") == "tool":
            tool_name = parsed.get("tool")
            args      = parsed.get("args", {})
            tools_used.append(tool_name)

            try:
                result = TOOLS[tool_name](**args) if tool_name in TOOLS else {"error": f"Unknown tool: {tool_name}"}
            except Exception as e:
                result = {"error": str(e)}

            obs = json.dumps(result, ensure_ascii=False)[:4000]
            messages.append({"role": "user", "content": f"Observation: {obs}"})

        steps += 1

    return {"answer": "", "sources": [], "tools_used": tools_used, "steps": steps}

In [6]:
# Cell D — Supervisor agent

SUPERVISOR_SYSTEM = """You are a supervisor coordinating sub-agents to answer financial document questions.

Synthesize sub-agent findings into one precise answer.

Return ONLY valid JSON:
{{
  "answer": "final answer",
  "company": "company name",
  "fiscal_year": "year",
  "documents_used": ["doc1"],
  "pages_used": ["p1-p3"],
  "document_extracts": ["key extract 1"],
  "extract_types": ["table", "narrative"],
  "confidence": 0.0-1.0
}}"""

SUB_AGENT_STRATEGIES = [
    "Search for exact figures, KPIs, and numeric data mentioned in the question.",
    "Search for narrative context and qualitative explanations around the topic.",
    "Search for tables and structured data related to the question.",
]

def run_supervisor(question: str, company: str, question_type: str,
                   fiscal_year: str = None, n_sub_agents: int = 3) -> dict:

    sub_agent_results = []
    agents_called     = []
    all_tools_used    = []

    with ThreadPoolExecutor(max_workers=n_sub_agents) as ex:
        futures = {
            ex.submit(run_sub_agent, question, company, SUB_AGENT_STRATEGIES[i]): i
            for i in range(n_sub_agents)
        }
        for f in as_completed(futures):
            idx    = futures[f]
            result = f.result()
            sub_agent_results.append((idx, result))
            agents_called.append(f"sub_agent_{idx}")
            all_tools_used.extend(result.get("tools_used", []))

    sub_agent_results.sort(key=lambda x: x[0])

    findings = "\n\n".join([
        f"Sub-agent {i} (strategy: {SUB_AGENT_STRATEGIES[i]}):\n"
        f"Answer: {r['answer']}\nSources: {json.dumps(r['sources'])}"
        for i, r in sub_agent_results
    ])

    messages = [
        {"role": "system", "content": SUPERVISOR_SYSTEM},
        {"role": "user",   "content":
            f"Company: {company}\nFiscal Year: {fiscal_year or 'unknown'}\n"
            f"Question Type: {question_type}\nQuestion: {question}\n\n"
            f"Sub-agent Findings:\n{findings}"}
    ]

    response, latency_ms = timed_completion(
        messages, n=MODEL_N,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}} if MODEL_N == 1 else {}
    )
    raw = strip_thinking(response.choices[0].message.content)
    raw = re.sub(r"```json|```", "", raw).strip()

    try:
        result = json.loads(raw)
    except json.JSONDecodeError:
        result = {"answer": raw, "documents_used": [], "pages_used": [],
                  "document_extracts": [], "extract_types": [], "confidence": 0.0}

    result["agents_called"]         = agents_called
    result["tools_used"]            = list(set(all_tools_used))
    result["supervisor_latency_ms"] = latency_ms
    result["sub_agent_results"]     = [r for _, r in sub_agent_results]

    return result

In [7]:
# Cell E — LLM Judge

JUDGE_SYSTEM = """You are an expert evaluator for financial document QA.

Score the predicted answer against ground truth.

Return ONLY valid JSON:
{{
  "score": 0-10,
  "reasoning": "brief explanation",
  "correct_facts": ["..."],
  "missing_facts": ["..."],
  "hallucinations": ["..."]
}}

10=exact, 7-9=mostly correct, 4-6=partial, 1-3=mostly wrong, 0=hallucinated."""

def run_judge(question: str, ground_truth: str, predicted: str, question_type: str) -> dict:
    messages = [
        {"role": "system", "content": JUDGE_SYSTEM},
        {"role": "user",   "content":
            f"Question Type: {question_type}\nQuestion: {question}\n"
            f"Ground Truth: {ground_truth}\nPredicted: {predicted}"}
    ]
    response, latency_ms = timed_completion(
        messages, n=MODEL_N,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}} if MODEL_N == 1 else {}
    )
    raw = strip_thinking(response.choices[0].message.content)
    raw = re.sub(r"```json|```", "", raw).strip()
    try:
        result = json.loads(raw)
    except json.JSONDecodeError:
        result = {"score": 0, "reasoning": raw, "correct_facts": [],
                  "missing_facts": [], "hallucinations": []}
    result["latency_ms"] = latency_ms
    return result

In [8]:
# Cell F — Load questions
# Expected columns: company, fiscal_year, question_id, question,
#                   question_type (LR/PE/NR), answer, perturbation_type, perturbation_level

QUESTIONS_PATH = PROJECT_ROOT / "datasets" / "cfb_perturbed.csv"
df_questions = pd.read_csv(QUESTIONS_PATH)
print(f"Questions loaded: {len(df_questions)}")

df_questions = df_questions.rename(columns={
    "Company's name":     "company",
    "Fiscal year":        "fiscal_year",
    "Question ID":        "question_id",
    "Question":           "question",
    "Type of question":   "question_type",
    "Answer":             "answer",
    "Documents":          "documents_src",
    "Pages":              "pages_src",
    "Document extracts":  "document_extracts_src",
    "Extract type":       "extract_type_src",
    "perturbation_type":  "perturbation_type",
    "perturbation_level": "perturbation_level",
})

print(df_questions.columns.tolist())
df_questions.head()

Questions loaded: 6270
['company', 'fiscal_year', 'question_id', 'question', 'question_type', 'answer', 'documents_src', 'pages_src', 'document_extracts_src', 'extract_type_src', 'perturbation_type', 'perturbation_level']


,company,fiscal_year,question_id,question,question_type,answer,documents_src,pages_src,document_extracts_src,extract_type_src,perturbation_type,perturbation_level
0,Ali Baba Group,2024,Q1,According to the company's Disclosure from FY2...,LR,According to the company's statement for fisca...,"2024 Alibaba Group Environmental, Social and G...",doc1{page7},doc1{\nRestoring Our Green Planet\nAddressing ...,text,none,0
1,Ali Baba Group,2024,Q2,What drove carbon intensity change as of the F...,PE,Ali Baba Group carbon intensity dropped in FY2...,"2024 Alibaba Group Environmental, Social and G...","doc1{page10, page11}",doc1{Emissions reduction from our own operatio...,text,none,0
2,Ali Baba Group,2024,Q3,Does the company have a decarbonization trajec...,LR,"\nNo, the company has several projects to deca...","2024 Alibaba Group Environmental, Social and G...","doc1{page31, page37, page157 }",doc1{We dug deep into four major scenarios for...,text,none,0
3,Ali Baba Group,2024,Q4,Does the company operate in a high emitting se...,LR,"Yes, Alibaba Group operates in the logistics s...","2024 Alibaba Group Environmental, Social and G...",doc1{page29},doc1{Reducing emissions in the logistics secto...,text,none,0
4,Ali Baba Group,2024,Q6,Has the company identified significant decarbo...,PE,"Yes, Alibaba Group has identified several sign...","2024 Alibaba Group Environmental, Social and G...","doc1{page26, page31, page33, page34}",doc1{Improving efficiency through technologies...,text,none,0


In [ ]:
# Cell G — Main pipeline loop

def process_question(row: dict) -> dict:
    company       = row["company"]
    fiscal_year   = str(row.get("fiscal_year", ""))
    question_id   = row["question_id"]
    question      = row["question"]
    question_type = row["question_type"]
    ground_truth  = row["answer"]
    perturb_type  = row.get("perturbation_type", "")
    perturb_level = row.get("perturbation_level", "")

    supervisor_result = run_supervisor(
        question=question,
        company=company,
        question_type=question_type,
        fiscal_year=fiscal_year,
        n_sub_agents=3,
    )

    predicted = supervisor_result.get("answer", "")

    judge_result = run_judge(
        question=question,
        ground_truth=ground_truth,
        predicted=predicted,
        question_type=question_type,
    )

    return {
        "company":               company,
        "fiscal_year":           fiscal_year,
        "question_id":           question_id,
        "question":              question,
        "question_type":         question_type,
        "answer":                predicted,
        "ground_truth":          ground_truth,
        "documents":             json.dumps(supervisor_result.get("documents_used", [])),
        "pages":                 json.dumps(supervisor_result.get("pages_used", [])),
        "document_extracts":     json.dumps(supervisor_result.get("document_extracts", [])),
        "extract_type":          json.dumps(supervisor_result.get("extract_types", [])),
        "perturbation_type":     perturb_type,
        "perturbation_level":    perturb_level,
        # Agent meta
        "agents_called":         json.dumps(supervisor_result.get("agents_called", [])),
        "tools_used":            json.dumps(supervisor_result.get("tools_used", [])),
        "judge_score":           judge_result.get("score"),
        "judge_reasoning":       judge_result.get("reasoning"),
        "correct_facts":         json.dumps(judge_result.get("correct_facts", [])),
        "missing_facts":         json.dumps(judge_result.get("missing_facts", [])),
        "hallucinations":        json.dumps(judge_result.get("hallucinations", [])),
        "supervisor_confidence": supervisor_result.get("confidence"),
        "supervisor_latency_ms": supervisor_result.get("supervisor_latency_ms"),
        "judge_latency_ms":      judge_result.get("latency_ms"),
    }


QA_WORKERS = 12  # tune to rate limits
results    = []

rows = df_questions.to_dict("records")
print(rows[0])
with ThreadPoolExecutor(max_workers=QA_WORKERS) as executor:
    futures = {executor.submit(process_question, row): row["question_id"] for row in rows}
    for f in tqdm(as_completed(futures), total=len(futures), desc="Questions"):
        try:
            results.append(f.result())
        except Exception as e:
            print(f"Error on {futures[f]}: {e}")

print(f"Done. {len(results)} results.")

{'company': 'Ali Baba Group', 'fiscal_year': 2024, 'question_id': 'Q1', 'question': "According to the company's Disclosure from FY2024, which topics have been assessed to be material?", 'question_type': 'LR', 'answer': 'According to the company\'s statement for fiscal 2023, the following topics are considered material: \nE : "restoring our green planet : Addressing major environmental issues such as climate change.", \nS : "building trust : Building corporate trust by establishing an effective,transparent, and sound governance system; building social trust with ethical technology, and protecting user privacy and data security.", "facilitating participatory philanthropy : Fostering a culture of participatory philanthropy by integrating community resources with creative platform innovations.", "supporting our people : Building a people-first culture that offers an equal, inclusive, and dignified environment in which every employee can grow and develop.", "enabling a sustainable digital l

Questions:   2%|▏         | 112/6270 [09:59<12:42:11,  7.43s/it]/var/folders/6c/859vmlp12zq4chp42c36p6mw0000gn/T/ipykernel_1194/206845853.py:19: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df_extraction["document"].str.contains(document, case=False, na=False) &
Questions:   2%|▏         | 115/6270 [10:06<7:00:40,  4.10s/it] /var/folders/6c/859vmlp12zq4chp42c36p6mw0000gn/T/ipykernel_1194/206845853.py:19: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df_extraction["document"].str.contains(document, case=False, na=False) &
Questions:   2%|▏         | 116/6270 [10:09<6:38:02,  3.88s/it]/var/folders/6c/859vmlp12zq4chp42c36p6mw0000gn/T/ipykernel_1194/206845853.py:19: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df_extraction["

In [ ]:
# Cell H — Save results

RESULTS_DIR = PROJECT_ROOT / "datasets" / "experiments" / "agent" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

timestamp  = datetime.now().strftime("%Y%m%d_%H%M%S")
df_results = pd.DataFrame(results)
df_results.to_csv(RESULTS_DIR / f"qa_results_{timestamp}.csv", index=False)

print(f"Saved: {len(df_results)} rows")
print(f"Avg judge score: {df_results['judge_score'].mean():.2f}")
print(df_results.groupby("question_type")["judge_score"].mean())

QA results:     0 rows
Agent metrics:  0 rows


KeyError: 'judge_score'